# P1 — What is over-squashing?

A GNN updates each node by mixing in its neighbours, one hop at a time. When a node needs information from
*far away* and the graph has a **bottleneck**, a huge amount of distant information is crammed into one
fixed-size vector. That cramming is **over-squashing**.

**Figures:** (1) the bottleneck graph, (2) how messages explode with depth, (3) the multiplicity heatmap.

In [ ]:
import torch, numpy as np
from oversquash import viz
from oversquash.data_bottleneck import make_bottleneck_graph
from oversquash.ideal_bridge import walk_operators
K, M = 4, 3

## Figure 1 — the bottleneck

Sources (blue) feed through a narrow middle (yellow) into a single target (green). Every path crosses the
narrow part.

In [ ]:
data = make_bottleneck_graph(K, M, 2, torch.Generator().manual_seed(0))
viz.draw_bottleneck(data, K, M, title='sources -> bottleneck -> target')

## Figure 2 — messages explode with depth

Count the walks of length `depth+1` reaching the target. A normal GNN must compress *all* of them into the
target's one vector. Watch the count blow up as the graph gets deeper (note the log scale).

In [ ]:
depths = [1,2,3,4]; msgs = []
for d in depths:
    dd = make_bottleneck_graph(K, M, d, torch.Generator().manual_seed(0))
    raw, _ = walk_operators(dd.edge_index.numpy(), dd.num_nodes, max_length=d+1)
    t = int(dd.root_mask.nonzero()[0]); msgs.append(int(raw[d][:, t].sum()))
viz.plot_message_explosion(depths, msgs, 'messages squashed into one vector', 'depth', 'messages (log)')

## Figure 3 — where the messages come from

The multiplicity matrix at the deepest range: row = target, column = source, colour = number of walks. The
bright column into the target node is the pile-up that gets squashed.

In [ ]:
deep = make_bottleneck_graph(K, M, 3, torch.Generator().manual_seed(0))
raw, _ = walk_operators(deep.edge_index.numpy(), deep.num_nodes, max_length=4)
viz.plot_multiplicity_heatmap(raw[3], 'walk multiplicity at range 4 (A^4)')
print('The path count grows ~K*M^d, but the target vector stays the same size. On to P2.')